# 04 — Many Models: Per-Series Baselines with `applyInPandas`

This notebook trains per-series baseline models (**SeasonalNaive**, **AutoETS**, and **Croston**) across all 30,490 M5 series using Spark's `groupBy("id").applyInPandas`.

- **Step 1:** `%pip install statsforecast` (must be run first as it restarts the Python process).
- **Step 2:** Register `src.baselines` code with `cloudpickle.register_pickle_by_value` to ensure workers receive the full implementation.
- **Step 3:** Benchmark execution time on a single store (`CA_1`, ~3k series) to verify scaling and quota.
- **Step 4:** Run across the full 30,490 series and persist to Delta table `workspace.m5.baselines`.

In [0]:
# The first cell must be the install — %pip restarts Python
%pip install statsforecast

In [0]:
import sys
import time

REPO = "/Workspace/Users/veysel/m5-forecasting-spark"  # adjust if your repo path differs
if REPO not in sys.path:
    sys.path.append(REPO)

from pyspark.sql import functions as F
import cloudpickle
import src.baselines

# Ship the module's bytecode to the Spark workers to prevent ModuleNotFoundError
cloudpickle.register_pickle_by_value(src.baselines)

from src.baselines import fit_baselines
from src.config import CUTOFFS

# Exclude TUNE fold for baselines to save 25% compute (only F1, F2, F3 needed for comparison)
BASELINE_CUTOFFS = {k: v for k, v in CUTOFFS.items() if k != 'TUNE'}

SCHEMA = "id string, fold string, d_num int, model string, yhat double, lo80 double, hi80 double"

def _fit(pdf):
    return fit_baselines(pdf, BASELINE_CUTOFFS)

base = spark.table("workspace.m5.base").select("id", "d_num", "sales", "store_id")
print(f"Loaded base table: {base.count():,} rows across {base.select('id').distinct().count():,} distinct series")

### Benchmark on CA_1 (~3,049 series)

Test on store `CA_1` first to verify the execution and extrapolate the runtime for all 10 stores.

In [0]:
t0 = time.time()
(
    base.filter("store_id = 'CA_1'")
    .drop("store_id")
    .repartition(32, "id")
    .groupBy("id")
    .applyInPandas(_fit, schema=SCHEMA)
    .write.mode("overwrite")
    .saveAsTable("workspace.m5.baselines_ca1")
)
elapsed_ca1 = time.time() - t0
print(f"CA_1 (~3k series) completed in: {elapsed_ca1:.0f}s ({elapsed_ca1 / 60:.2f} min)")
print(f"Estimated time for all 30k series: ~{(elapsed_ca1 * 10) / 60:.1f} min")

### Inspect CA_1 Baseline Results

In [0]:
ca1_res = spark.table("workspace.m5.baselines_ca1")
display(ca1_res.limit(20))
print("Row count:", ca1_res.count())
print("Models present:", [r.model for r in ca1_res.select("model").distinct().collect()])
print("Folds present:", [r.fold for r in ca1_res.select("fold").distinct().collect()])

### Run on Full Set (30,490 series)

> **Note:** If cluster time / quota is constrained, consider dropping `'TUNE'` from `CUTOFFS` or using the last 365 days for AutoETS before running full 30k series.

In [0]:
t_full = time.time()
(
    base.drop("store_id")
    .repartition(128, "id")
    .groupBy("id")
    .applyInPandas(_fit, schema=SCHEMA)
    .write.mode("overwrite")
    .saveAsTable("workspace.m5.baselines")
)
elapsed_full = time.time() - t_full
print(f"Full 30,490 series baselines completed in: {elapsed_full:.0f}s ({elapsed_full / 60:.2f} min)")

### Validation of Full Baselines Table

In [0]:
baselines_df = spark.table("workspace.m5.baselines")
print(f"Total baseline forecast rows: {baselines_df.count():,}")
display(baselines_df.groupBy("fold", "model").count())